In [ ]:
from google.colab import drive
import os
import gc
import torch
import polars as pl
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from tqdm import tqdm


drive.mount('/content/drive')

INPUT_FILE = "/content/drive/MyDrive/capstone/data/temp_transcripts.parquet"
OUTPUT_DIR = "/content/drive/MyDrive/Post-Earnings/finbert_tx/"
os.makedirs(OUTPUT_DIR, exist_ok=True)


In [ ]:
print("Files in Drive:")
for root, dirs, files in os.walk("/content/drive/MyDrive/"):
    for file in files:
        print(os.path.join(root, file))

In [ ]:


INFERENCE_BATCH_SIZE = 64

def run_colab_inference():
    device = torch.device("cuda")

    tokenizer = AutoTokenizer.from_pretrained("ProsusAI/finbert")
    model = AutoModelForSequenceClassification.from_pretrained("ProsusAI/finbert")
    model.to(device).half().eval()

    tx_lz = (
        pl.scan_parquet(INPUT_FILE)
        .explode("transcript")
        .unnest("transcript")
        .select(["symbol", "reportedDate", pl.col("speaker"), pl.col("content").alias("text")])
        .filter(pl.col("speaker") != "Operator")
    )

    all_symbols = tx_lz.select("symbol").unique().collect()["symbol"].to_list()

    for symbol in all_symbols:
        if os.path.exists(os.path.join(OUTPUT_DIR, f"{symbol}.parquet")):
            continue

        print(f"Processing {symbol}")
        symbol_data = tx_lz.filter(pl.col("symbol") == symbol).collect()
        if symbol_data.is_empty(): continue

        texts = symbol_data["text"].to_list()
        all_predictions = []

        with torch.no_grad():
            for i in tqdm(range(0, len(texts), INFERENCE_BATCH_SIZE), desc=f"  Inference for {symbol}"):
                batch_texts = texts[i : i + INFERENCE_BATCH_SIZE]
                encoding = tokenizer(
                    batch_texts, add_special_tokens=True, max_length=512,
                    padding='max_length', truncation=True, return_tensors='pt'
                ).to(device)

                outputs = model(**encoding)
                probs = torch.nn.functional.softmax(outputs.logits, dim=1)
                all_predictions.extend(probs.cpu().numpy())

        symbol_data = symbol_data.with_columns(pl.Series("sentiment_probs", all_predictions))
        symbol_data.write_parquet(os.path.join(OUTPUT_DIR, f"{symbol}.parquet"))

        del symbol_data, all_predictions
        gc.collect()
        torch.cuda.empty_cache()